In [2]:
import matplotlib.pyplot as plt
import pandas as pd

# Load CSV
df = pd.read_csv("cboddick_project_summary.csv", sep=";")

df["date"] = pd.to_datetime(df["time"], unit="s")
df["Month"] = df["date"].dt.to_period("M")

for project in df["project_wocid"].unique():

    # Get commits for only this project
    project_df = df[df["project_wocid"] == project]

    # Count commits by month
    monthly = project_df.groupby("Month").size()

    # Create every month from first commit through last commit
    all_months = pd.period_range(
        start=monthly.index.min(),
        end=monthly.index.max(),
        freq="M"
    )

    # Fill months with no commits with 0
    monthly = monthly.reindex(all_months, fill_value=0)

    # Turn into dataframe
    timeseries = monthly.reset_index()
    timeseries.columns = ["Month", "#Commits"]
    timeseries["Month"] = timeseries["Month"].astype(str)

    # Save semicolon-separated CSV
    csv_name = f"cboddick_commits_timeseries_{project}.csv"
    timeseries.to_csv(csv_name, sep=";", index=False)

    # Make plot
    plt.figure(figsize=(10, 5))
    plt.plot(
        timeseries["Month"],
        timeseries["#Commits"],
        linestyle="-"
    )

    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(project)
    plt.grid(True)

    # Don't display every single month label if there are many months
    step = max(1, len(timeseries) // 12)
    plt.xticks(
        range(0, len(timeseries), step),
        timeseries["Month"].iloc[::step],
        rotation=45
    )

    plt.tight_layout()

    # Save at required resolution
    plot_name = f"cboddick_timeseries_{project}.png"
    plt.savefig(plot_name, dpi=300)
    plt.close()

    print("Created:", csv_name)
    print("Created:", plot_name)




Created: cboddick_commits_timeseries_maria-korosteleva_garment-pattern-estimation.csv
Created: cboddick_timeseries_maria-korosteleva_garment-pattern-estimation.png
Created: cboddick_commits_timeseries_tf-encrypted_tf-encrypted.csv
Created: cboddick_timeseries_tf-encrypted_tf-encrypted.png
Created: cboddick_commits_timeseries_baggepinnen_spectraldistances.jl.csv
Created: cboddick_timeseries_baggepinnen_spectraldistances.jl.png
Created: cboddick_commits_timeseries_massimocimmino_pygfunction.csv
Created: cboddick_timeseries_massimocimmino_pygfunction.png
Created: cboddick_commits_timeseries_the-virtual-brain_tvb-root.csv
Created: cboddick_timeseries_the-virtual-brain_tvb-root.png
Created: cboddick_commits_timeseries_xtra-computing_thundergbm.csv
Created: cboddick_timeseries_xtra-computing_thundergbm.png
Created: cboddick_commits_timeseries_itowns_itowns.csv
Created: cboddick_timeseries_itowns_itowns.png
Created: cboddick_commits_timeseries_serock3_master_thesis_qec_simulation.csv
Created:

In [2]:
import pandas as pd

df_commit_data = pd.read_csv("df_commit_data.csv")

# Convert Unix timestamps to months
df_commit_data["date"] = pd.to_datetime(df_commit_data["author_time"], unit="s")
df_commit_data["Month"] = df_commit_data["date"].dt.to_period("M")

gap_results = []

for project in df_commit_data["project"].unique():

    project_df = df_commit_data[df_commit_data["project"] == project].copy()

    # Count commits in each month
    monthly = project_df.groupby("Month").size()

    # Include EVERY month from first to last commit
    all_months = pd.period_range(
        start=monthly.index.min(),
        end=monthly.index.max(),
        freq="M"
    )

    monthly = monthly.reindex(all_months, fill_value=0)

    # Find all continuous zero-commit gaps
    gaps = []
    current_start = None
    current_length = 0

    for month, count in monthly.items():

        if count == 0:

            if current_start is None:
                current_start = month

            current_length += 1

        else:

            if current_start is not None:
                gaps.append({
                    "start": current_start,
                    "end": month - 1,
                    "length": current_length
                })

                current_start = None
                current_length = 0

    # In case timeline ends with zero months
    if current_start is not None:
        gaps.append({
            "start": current_start,
            "end": monthly.index[-1],
            "length": current_length
        })

    # Find longest gap
    if gaps:
        longest_gap = max(gaps, key=lambda x: x["length"])

        gap_start = str(longest_gap["start"])
        gap_end = str(longest_gap["end"])
        gap_length = longest_gap["length"]

        # Commits AFTER the longest gap ends
        commits_after_gap = monthly[
            monthly.index > longest_gap["end"]
        ].sum()

    else:
        gap_start = ""
        gap_end = ""
        gap_length = 0
        commits_after_gap = 0

    # Count gaps that lasted AT LEAST 3 months
    number_of_gaps = sum(
        1 for gap in gaps if gap["length"] >= 3
    )

    gap_results.append({
        "project": project,
        "LongestGapStart": gap_start,
        "LongestGapEnd": gap_end,
        "LongestGapLength": gap_length,
        "NumCommitsAfterLastGap": int(commits_after_gap),
        "NumberOfGapsInTimeline": number_of_gaps
    })

gap_df = pd.DataFrame(gap_results)

gap_df

project_stats = df_commit_data.groupby("project").agg(
    number_of_commits=("commit", "nunique"),
    number_of_authors=("author", "nunique"),
    max_time=("author_time", "max"),
    min_time=("author_time", "min")
).reset_index()

project_stats

github_data = pd.DataFrame([
    {
        "project": "maria-korosteleva_garment-pattern-estimation",
        "number of stars": 181,
        "number of forks": 21,
        "last commit date": "2024-12-23"
    },
    {
        "project": "tf-encrypted_tf-encrypted",
        "number of stars": 1200,
        "number of forks": 211,
        "last commit date": "2024-09-25"
    },
    {
        "project": "baggepinnen_spectraldistances.jl",
        "number of stars": 49,
        "number of forks": 5,
        "last commit date": "2026-01-15"
    },
    {
        "project": "massimocimmino_pygfunction",
        "number of stars": 72,
        "number of forks": 26,
        "last commit date": "2025-08-04"
    },
    {
        "project": "the-virtual-brain_tvb-root",
        "number of stars": 190,
        "number of forks": 151,
        "last commit date": "2026-09-23"
    },
    {
        "project": "xtra-computing_thundergbm",
        "number of stars": 715,
        "number of forks": 88,
        "last commit date": "2025-03-19"
    },
    {
        "project": "itowns_itowns",
        "number of stars": 1300,
        "number of forks": 317,
        "last commit date": "2026-09-14"
    },
    {
        "project": "serock3_master_thesis_qec_simulation",
        "number of stars": 5,
        "number of forks": 0,
        "last commit date": "2024-09-07"
    },
    {
        "project": "raysect_source",
        "number of stars": 112,
        "number of forks": 25,
        "last commit date": "2025-08-25"
    },
    {
        "project": "nimble-dev_nimble",
        "number of stars": 198,
        "number of forks": 27,
        "last commit date": "2026-08-31"
    }
])

final_stats = project_stats.merge(
    github_data,
    on="project",
    how="left"
)

final_stats = final_stats.merge(
    gap_df,
    on="project",
    how="left"
)

final_stats



,project,number_of_commits,number_of_authors,max_time,min_time,number of stars,number of forks,last commit date,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
0,baggepinnen_spectraldistances.jl,759,7,1776126985,1568771532,49,5,2026-01-15,2023-02,2025-06,29,29,3
1,itowns_itowns,9190,142,1762275571,1443101686,1300,317,2026-09-14,,,0,0,0
2,maria-korosteleva_garment-pattern-estimation,1378,11,1736211317,1582260169,181,21,2024-12-23,2023-12,2024-11,12,2,3
3,massimocimmino_pygfunction,1385,22,1771443588,1502741635,72,26,2025-08-04,2018-04,2018-10,7,1136,7
4,nimble-dev_nimble,9730,73,1762381653,1402585122,198,27,2026-08-31,,,0,0,0
5,raysect_source,2366,33,1758534567,1376432727,112,25,2025-08-25,2024-03,2024-09,7,143,4
6,serock3_master_thesis_qec_simulation,496,6,1639215098,1611075452,5,0,2024-09-07,2021-11,2021-11,1,1,0
7,tf-encrypted_tf-encrypted,3165,49,1727242358,1521656591,1200,211,2024-09-25,2023-11,2024-08,10,2,4
8,the-virtual-brain_tvb-root,10306,136,1776180337,1401877909,190,151,2026-09-23,2018-07,2018-11,5,7764,1
9,xtra-computing_thundergbm,683,24,1742375195,1453180646,715,88,2025-03-19,2024-02,2025-02,13,2,7


In [4]:
activity_patterns = {
    "maria-korosteleva_garment-pattern-estimation": "declining",
    "tf-encrypted_tf-encrypted": "declining",
    "baggepinnen_spectraldistances.jl": "cyclical",
    "massimocimmino_pygfunction": "irregular",
    "the-virtual-brain_tvb-root": "steady",
    "xtra-computing_thundergbm": "declining",
    "itowns_itowns": "declining",
    "serock3_master_thesis_qec_simulation": "cyclical",
    "raysect_source": "cyclical",
    "nimble-dev_nimble": "steady"
}

final_stats["ActivityPattern"] = final_stats["project"].map(activity_patterns)

final_stats.to_csv(
    "cboddick_project_stats.csv",
    sep=";",
    index=False
)

final_stats

,project,number_of_commits,number_of_authors,max_time,min_time,number of stars,number of forks,last commit date,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,baggepinnen_spectraldistances.jl,759,7,1776126985,1568771532,49,5,2026-01-15,2023-02,2025-06,29,29,3,cyclical
1,itowns_itowns,9190,142,1762275571,1443101686,1300,317,2026-09-14,,,0,0,0,declining
2,maria-korosteleva_garment-pattern-estimation,1378,11,1736211317,1582260169,181,21,2024-12-23,2023-12,2024-11,12,2,3,declining
3,massimocimmino_pygfunction,1385,22,1771443588,1502741635,72,26,2025-08-04,2018-04,2018-10,7,1136,7,irregular
4,nimble-dev_nimble,9730,73,1762381653,1402585122,198,27,2026-08-31,,,0,0,0,steady
5,raysect_source,2366,33,1758534567,1376432727,112,25,2025-08-25,2024-03,2024-09,7,143,4,cyclical
6,serock3_master_thesis_qec_simulation,496,6,1639215098,1611075452,5,0,2024-09-07,2021-11,2021-11,1,1,0,cyclical
7,tf-encrypted_tf-encrypted,3165,49,1727242358,1521656591,1200,211,2024-09-25,2023-11,2024-08,10,2,4,declining
8,the-virtual-brain_tvb-root,10306,136,1776180337,1401877909,190,151,2026-09-23,2018-07,2018-11,5,7764,1,steady
9,xtra-computing_thundergbm,683,24,1742375195,1453180646,715,88,2025-03-19,2024-02,2025-02,13,2,7,declining


## Interpretation

### maria-korosteleva/Garment-Pattern-Estimation
- Pattern: Declining
- Explaination: This project shows heavy activity between Feburary of 2020 and March of 2022. After this there is a huge drop in activity with little to no activity from March 2022 onward.
- Number of Gaps: 3

### tf-encrypted/tf-encrypted
- Pattern: Declining
- Explaination: The project shows heavy activity starting in March 2018 and it slowly declines until September 2020 where there is a long stretch of little to no activity. There is a small spike in March 2022 but stays little to none besides that. 
- Number of Gaps: 4

### baggepinnen/SpectralDistances.jl
- Pattern: Cyclical
- Explaination: This project has three small spikes in ativity. late 2019/early 2020, mid 2022, and mid 2025. Besides these three spikes there is little to no activity.
- Number of Gaps: 3

### MassimoCimmino/pygfunction
- Pattern: Irregular
- Explaination: This project shows irregular activity throughout its lifetime. It follows no patterns as it has random periods of high, low, and no activity at all.
- Number of Gaps: 7

### the-virtual-brain/tvb-root
- Pattern: Steady
- Explaination: This project shows steady activity throughout its lifetime. It has constant activity besides a hight spile of commits througout 2019. Besides that it is similar throughout.
- Number of Gaps: 1

### Xtra-Computing/thundergbm
- Pattern: Declining
- Explaination: This project shows heavy activity at the beggining of its life with small gaps here and there. It takes a nosedive in early 2020 where the activity drops. There is a small spike in mid 2023. 
- Number of Gaps: 7

### iTowns/itowns
- Pattern: Declining
- Explaination: This project has very steady activity with no real gaps, however it is on a slow and steady decline. starting with spikes around 300 commits and slowly lowering to <100 towards the end.
- Number of Gaps: 0

### Serock3/Master-Thesis-QEC-simulation
- Pattern: Cyclical
- Explaination: This project has only 1 year of commit history, which made it a bit difficult to classify. But, I chose cyclical because it starts low in January, raises in March-May, Lowers in June, Raises in August, then lowers for the rest of the year. becuaes of this incline and decline cycle I went with cyclical.
- Number of Gaps: 0

### raysect/source
- Pattern: Cyclical
- Explaination: While this project does have a very steady overall strucure, it has dips and rises throughout. It averages around 20-30 commits, but it has about 5 spikes spread evenly throuhout the lifecycle.
- Number of Gaps: 4

### nimble-dev/nimble
- Pattern: Steady
- Explaination: This project has steady commits throughout with a few spikes early on compared to later, but is quite steady throuhgout. 
- Number of Gaps: 0

In [5]:
# Load the files
import pandas as pd

summary = pd.read_csv(
    "cboddick_project_summary.csv",
    sep=";"
)

stats = pd.read_csv(
    "cboddick_project_stats.csv",
    sep=";"
)

summary.head()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


,project_wocid,commit_sha1,author,time,commit message
0,maria-korosteleva_garment-pattern-estimation,005873aebe765beedd8bb2cda3184f8c89279d21,Maria Korosteleva <mariako@kaist.ac.kr>,1617698599,recording stats fix\n
1,maria-korosteleva_garment-pattern-estimation,00e5150fb1212c26300fffb603049b2d3e3c2c96,Maria Korosteleva <mariako@kaist.ac.kr>,1590999070,skirt upgrade v0.2\n
2,maria-korosteleva_garment-pattern-estimation,013606d395963877b82b041e0b44afc36b4ec05e,mariako <mariako@kaist.ac.kr>,1621824387,transfer segmentation labels during scan imita...
3,maria-korosteleva_garment-pattern-estimation,01384216aa713d0051f13e2db0f82a1a8b23a89c,Maria Korosteleva <mariako@kaist.ac.kr>,1593576105,add feature sizes info to config\n
4,maria-korosteleva_garment-pattern-estimation,01567bf034a1e90c3525f395e6f8a9084b331035,Maria Korosteleva <mariako@kaist.ac.kr>,1606703775,"Revert ""eval stitch sides loss right away""\n\n..."


In [6]:
summary["datetime"] = pd.to_datetime(
    summary["time"],
    unit="s"
)

summary["Month"] = summary["datetime"].dt.to_period("M")

gap_commit_rows = []

for _, row in stats.iterrows():

    project = row["project"]
    gap_start = row["LongestGapStart"]
    gap_end = row["LongestGapEnd"]

    project_commits = summary[
        summary["project_wocid"] == project
    ].copy()

    project_commits = project_commits.sort_values("time")

    # Projects with no inactivity gap
    if pd.isna(gap_start) or pd.isna(gap_end) or gap_start == "" or gap_end == "":
        print(project, "- no inactivity gap")
        continue

    gap_start = pd.Period(gap_start, freq="M")
    gap_end = pd.Period(gap_end, freq="M")

    # Last 10 commits before longest gap
    pre = project_commits[
        project_commits["Month"] < gap_start
    ].tail(10).copy()

    pre["pre/post"] = "pre"

    # First 10 commits after longest gap
    post = project_commits[
        project_commits["Month"] > gap_end
    ].head(10).copy()

    post["pre/post"] = "post"

    gap_commit_rows.append(pre)
    gap_commit_rows.append(post)

itowns_itowns - no inactivity gap
nimble-dev_nimble - no inactivity gap


In [7]:
gap_commits = pd.concat(
    gap_commit_rows,
    ignore_index=True
)

gap_commits = gap_commits[
    [
        "pre/post",
        "project_wocid",
        "commit_sha1",
        "author",
        "time",
        "commit message"
    ]
].copy()

gap_commits.columns = [
    "pre/post",
    "prj",
    "commit",
    "author",
    "time",
    "message"
]

gap_commits.to_csv(
    "cboddick_project_gap_commits.csv",
    sep=";",
    index=False
)

gap_commits.head()

gap_commits.groupby(
    ["prj", "pre/post"]
).size()



prj                                           pre/post
baggepinnen_spectraldistances.jl              post        10
                                              pre         10
maria-korosteleva_garment-pattern-estimation  post         2
                                              pre         10
massimocimmino_pygfunction                    post        10
                                              pre         10
raysect_source                                post        10
                                              pre         10
serock3_master_thesis_qec_simulation          post         1
                                              pre         10
tf-encrypted_tf-encrypted                     post         2
                                              pre         10
the-virtual-brain_tvb-root                    post        10
                                              pre         10
xtra-computing_thundergbm                     post         2
                              

In [8]:
for project in gap_commits["prj"].unique():

    print("\n" + "=" * 80)
    print(project)
    print("=" * 80)

    for period in ["pre", "post"]:

        print("\n", period.upper(), "GAP COMMITS")

        messages = gap_commits[
            (gap_commits["prj"] == project) &
            (gap_commits["pre/post"] == period)
        ]["message"]

        for i, message in enumerate(messages, 1):
            print(f"{i}. {message}")


baggepinnen_spectraldistances.jl

 PRE GAP COMMITS
1. up compat

2. fix docs build

3. add some missing methods

4. run CI on julia v1.8

5. build based on 0b3c45e

6. build based on eca2365

7. define Distances._result_type

8. build based on 91f2213

9. CompatHelper: bump compat for Hungarian to 0.7, (keep existing compat)

10. Merge ec09b12d1327509f7296f6eabe77144fc28946fc into 91f2213dfff36d61a381e4bb82b130ccea2ecbec


 POST GAP COMMITS
1. CompatHelper: bump compat for DSP to 0.8, (keep existing compat)

2. CompatHelper: bump compat for StatsBase to 0.34, (keep existing compat)

3. CompatHelper: bump compat for FillArrays to 1, (keep existing compat)

4. CompatHelper: add new compat entry for Statistics at version 1, (keep existing compat)

5. compat updates

6. bump version

7. Merge branch 'master' of github.com:baggepinnen/SpectralDistances.jl

8. build based on 74f4206

9. CompatHelper: bump compat for FillArrays to 1, (keep existing compat)

10. CompatHelper: add new compat e

In [9]:
before_themes = {
    "baggepinnen_spectraldistances.jl":
        "Dependency updates:Feature development",

    "itowns_itowns":
        "N/A active Project",

    "maria-korosteleva_garment-pattern-estimation":
        "Documentation updates:Dependency updates",

    "massimocimmino_pygfunction":
        "Bug fixes:Documentation updates",

    "nimble-dev_nimble":
        "N/A active Project",

    "raysect_source":
        "Feature development:Other",

    "serock3_master_thesis_qec_simulation":
        "Documentation updates:Feature development",

    "tf-encrypted_tf-encrypted":
        "Bug fixes:Documentation updates",

    "the-virtual-brain_tvb-root":
        "Documentation updates:Other",

    "xtra-computing_thundergbm":
        "Feature development:Bug fixes"
}

after_themes = {
    "baggepinnen_spectraldistances.jl":
        "Dependency updates:Release",

    "itowns_itowns":
        "N/A active Project",

    "maria-korosteleva_garment-pattern-estimation":
        "Bug fixes:Other",

    "massimocimmino_pygfunction":
        "Feature development:Documentation updates",

    "nimble-dev_nimble":
        "N/A active Project",

    "raysect_source":
        "Other:Dependency updates",

    "serock3_master_thesis_qec_simulation":
        "Documentation updates",

    "tf-encrypted_tf-encrypted":
        "Bug fixes",

    "the-virtual-brain_tvb-root":
        "Release:Bug fixes",

    "xtra-computing_thundergbm":
        "Documentation updates"
}

final_stats["BeforeThemes"] = final_stats["project"].map(before_themes)
final_stats["AfterThemes"] = final_stats["project"].map(after_themes)

final_stats.to_csv(
    "cboddick_project_stats.csv",
    sep=";",
    index=False
)

final_stats[
    ["project", "BeforeThemes", "AfterThemes"]
]

,project,BeforeThemes,AfterThemes
0,baggepinnen_spectraldistances.jl,Dependency updates:Feature development,Dependency updates:Release
1,itowns_itowns,N/A active Project,N/A active Project
2,maria-korosteleva_garment-pattern-estimation,Documentation updates:Dependency updates,Bug fixes:Other
3,massimocimmino_pygfunction,Bug fixes:Documentation updates,Feature development:Documentation updates
4,nimble-dev_nimble,N/A active Project,N/A active Project
5,raysect_source,Feature development:Other,Other:Dependency updates
6,serock3_master_thesis_qec_simulation,Documentation updates:Feature development,Documentation updates
7,tf-encrypted_tf-encrypted,Bug fixes:Documentation updates,Bug fixes
8,the-virtual-brain_tvb-root,Documentation updates:Other,Release:Bug fixes
9,xtra-computing_thundergbm,Feature development:Bug fixes,Documentation updates


In [1]:
import pandas as pd
import re
from collections import Counter
# load files

stats = pd.read_csv(
    "cboddick_project_stats.csv",
    sep=";"
)

summary = pd.read_csv(
    "cboddick_project_summary.csv",
    sep=";"
)

gap_commits = pd.read_csv(
    "cboddick_project_gap_commits.csv",
    sep=";"
)


gap_reasons = {

    "baggepinnen_spectraldistances.jl":
        "I dont see an obvious cause in the commit history. The project entered the gap after routine compatibility, CI, documentation, and small feature work. Therefore I belive he gap was most likely due to a small break between a version release and maintenance",

    "itowns_itowns":
        "N/A active Project",

    "maria-korosteleva_garment-pattern-estimation":
        "the pre-gap work had shifted mainly to README, installation, and requirements maintenance, which suggests it entered a gap due to no maintenance needed.",

    "massimocimmino_pygfunction":
        "The gap followed the version 1.1.0 release cycle and a sequence of bug fixes, which suggests a pause between development and release cycles.",

    "nimble-dev_nimble":
        "N/A active Project",

    "raysect_source":
        "The gap followed migration work and a v0.8.1 build fix, suggesting a temporary pause between development phases.",

    "serock3_master_thesis_qec_simulation":
        "The repository was created for a Master's thesis and a six-week continuation project. The inactivity period is probably because of completion of an academic project phase.",

    "tf-encrypted_tf-encrypted":
        "The pre-gap commits include compatibility and bug fixes. Multiple installation and compatibility problems were also reported around this period, suggesting reduced maintenance during a difficult compatibility phase.",

    "the-virtual-brain_tvb-root":
        "The surrounding commits indicate repository reorganization and migration work. This shows possible completion of the project before publishing. post-gap commits mention moving and renaming code before publishing fully on GitHub.",

    "xtra-computing_thundergbm":
        "The gap followed a heavy optimization and bug-fix phase, while the immediately post-gap activity consisted only of README updates, suggesting that core development had substantially slowed."
}


why_recovered = {

    "baggepinnen_spectraldistances.jl":
        "Activity restarted dependency updates, followed by compatibility maintenance, a version bump, and build-related commits. The gap seemed like more of a pause in development and was recovered after a routine break after a version release. ",

    "itowns_itowns":
        "N/A active Project",

    "maria-korosteleva_garment-pattern-estimation":
        "The project returned to address issue #11 by fixing prediction naming conventions, followed by a merge commit. While it did recover, it was for only a short period. Therefore, I belive it recovered shortly to fix some final issues before going inactive again.",

    "massimocimmino_pygfunction":
        "Development resumed with substantial feature work, including a networks package, a Network class, and new interface. This suggests it recovered gracefully and according to schedule after a release cycle break.",

    "nimble-dev_nimble":
        "N/A active Project",

    "raysect_source":
        "Development resumed with setup and build refactoring, dependency constraint updates, and a bug fix. This one again suggests a scheduled break between development cycles that recovered gracefully.",

    "serock3_master_thesis_qec_simulation":
        "The first available post-gap WoC commit was a small README spelling and grammar update, so the initial return was brief and documentation-focused. This suggests the brief return was to fix some small issues and typos before final submission of the thesis.",

    "tf-encrypted_tf-encrypted":
        "The project returned for a narrow bug fix. Only two post-gap commits were available, so the recovery was limited. I believe the recovery was a final unsuccessful attempt to fix complex bugs, before abandoning the project alltogether.",

    "the-virtual-brain_tvb-root":
        "Activity resumed during GitHub publication and repository integration work, including merges from TVB framework and library branches, Docker-related work, and bug fixes. This suggests the recovery was due to final touches and migration patches after completeting the bulk of the project and taking a break, but before publishing the work.",

    "xtra-computing_thundergbm":
        "The gap ended with two README updates. This suggest a short document related recovery, rather than a broad return to core feature development."
}


stats["HypothesizedGapReason"] = stats["project"].map(gap_reasons)
stats["WhyRecovered"] = stats["project"].map(why_recovered)


post_counts = (
    gap_commits[
        gap_commits["pre/post"] == "post"
    ]
    .groupby("prj")
    .size()
    .to_dict()
)


def has_recovered(row):

    gap_length = row["LongestGapLength"]

    if pd.isna(gap_length) or float(gap_length) == 0:
        return "N/A"

    if post_counts.get(row["project"], 0) > 0:
        return "Yes"

    return "No"


stats["HasRecovered"] = stats.apply(
    has_recovered,
    axis=1
)


def author_id(author):

    author = str(author).strip()

    match = re.search(
        r"<([^>]+)>",
        author
    )

    if match:
        return match.group(1).strip().lower()

    return author.lower()


def author_name(author):

    author = str(author).strip()

    if "<" in author:
        return author.split("<")[0].strip()

    return author


def who_recovered(project):

    project_rows = gap_commits[
        gap_commits["prj"] == project
    ]

    if project_rows.empty:
        return "N/A"

    pre_rows = project_rows[
        project_rows["pre/post"] == "pre"
    ]

    post_rows = project_rows[
        project_rows["pre/post"] == "post"
    ]

    if post_rows.empty:
        return "N/A"

    pre_map = {
        author_id(a): author_name(a)
        for a in pre_rows["author"].dropna()
    }

    post_map = {
        author_id(a): author_name(a)
        for a in post_rows["author"].dropna()
    }

    pre_ids = set(pre_map.keys())
    post_ids = set(post_map.keys())

    returning = post_ids & pre_ids
    new = post_ids - pre_ids

    parts = []

    if returning:

        returning_names = sorted({
            post_map.get(
                x,
                pre_map.get(x, x)
            )
            for x in returning
        })

        parts.append(
            "Returning: " +
            ", ".join(returning_names)
        )

    if new:

        new_names = sorted({
            post_map[x]
            for x in new
        })

        parts.append(
            "New: " +
            ", ".join(new_names)
        )

    if not parts:
        return "No identifiable post-gap authors"

    return "; ".join(parts)


stats["WhoRecovered"] = stats["project"].apply(
    who_recovered
)


# Projects with NO zero-commit inactivity gap
no_gap_mask = (
    stats["LongestGapLength"]
    .fillna(0)
    .astype(float)
    == 0
)

stats.loc[
    no_gap_mask,
    "HypothesizedGapReason"
] = "N/A active Project"

stats.loc[
    no_gap_mask,
    "HasRecovered"
] = "N/A"

stats.loc[
    no_gap_mask,
    "WhyRecovered"
] = "N/A"

stats.loc[
    no_gap_mask,
    "WhoRecovered"
] = "N/A"


if "last commit date" in stats.columns:

    gh_date_col = "last commit date"

elif "lastGHCommitDate" in stats.columns:

    gh_date_col = "lastGHCommitDate"

else:

    raise KeyError(
        "Could not find the GitHub last commit date column."
    )


stats["_gh_date"] = pd.to_datetime(
    stats[gh_date_col],
    errors="coerce"
)

cutoff = pd.Timestamp("2025-04-01")


stats["Currentstatus"] = stats["_gh_date"].apply(
    lambda date:
        "Inactive"
        if pd.notna(date) and date < cutoff
        else "Active"
)


def classify_theme(message):

    m = str(message).lower()

    # Automated bot contributions
    if any(x in m for x in [
        "dependabot",
        "compathelper",
        "[bot]",
        "github-actions",
        "pre-commit.ci",
        "renovate"
    ]):
        return "Automated bot contributions"

    # Security
    if any(x in m for x in [
        "security",
        "cve-",
        "vulnerability",
        "vulnerab",
        "exploit"
    ]):
        return "Security patches"

    # Documentation
    if any(x in m for x in [
        "readme",
        "documentation",
        " docs",
        "docs ",
        "docstring",
        "spelling",
        "grammar",
        "typo",
        "citation",
        "bibtex",
        "example script",
        "example notebook"
    ]):
        return "Documentation updates"

    # Dependencies / compatibility
    if any(x in m for x in [
        "dependency",
        "dependencies",
        "requirements",
        "requirement",
        "compat",
        "numpy version",
        "cython version",
        "install_requires",
        "pyproject",
        "package version",
        "upgrade package"
    ]):
        return "Dependency updates"

    # Releases
    if any(x in m for x in [
        "release",
        "bump version",
        "increment version",
        "new version",
        "version 0.",
        "version 1.",
        "version 2."
    ]):
        return "Release"

    # Bug fixes
    if any(x in m for x in [
        "fix",
        "bug",
        "error",
        "incorrect",
        "wrong",
        "crash",
        "regression",
        "issue #",
        "issue#",
        "exception"
    ]):
        return "Bug fixes"

    # Features / enhancements
    if any(x in m for x in [
        "add ",
        "added ",
        "implement",
        "support ",
        "feature",
        "create ",
        "initial ",
        "refactor",
        "optimiz",
        "improv",
        "new ",
        "interface",
        "class",
        "module",
        "network"
    ]):
        return "Feature development"

    return "Other"


def top_two_themes(messages):

    messages = list(messages)

    if len(messages) == 0:
        return "N/A"

    themes = [
        classify_theme(m)
        for m in messages
    ]

    counts = Counter(themes)

    # Tie breaker:
    # whichever theme appears earlier in the recent commit list
    first_position = {}

    for i, theme in enumerate(themes):

        if theme not in first_position:
            first_position[theme] = i

    ranked = sorted(
        counts.keys(),
        key=lambda t: (
            -counts[t],
            first_position[t]
        )
    )

    return ":".join(
        ranked[:2]
    )


summary = summary.sort_values(
    [
        "project_wocid",
        "time"
    ]
)


recent_themes = {}


for project in stats["project"]:

    recent = (
        summary[
            summary["project_wocid"] == project
        ]
        .sort_values("time")
        .tail(10)
    )

    recent_themes[project] = top_two_themes(
        recent["commit message"]
    )


stats["RecentThemes"] = stats["project"].map(
    recent_themes
)

# Now save everything to the stats file
# Remove helper column used only to calculate current status
if "_gh_date" in stats.columns:
    stats = stats.drop(columns=["_gh_date"])

# Save all new information back to project_stats
stats.to_csv(
    "cboddick_project_stats.csv",
    sep=";",
    index=False
)

print("Saved Part 3.2 Steps 1-3 to cboddick_project_stats.csv")

# Display the new columns so the results can be checked
display(
    stats[
        [
            "project",
            "HypothesizedGapReason",
            "HasRecovered",
            "WhyRecovered",
            "WhoRecovered",
            "Currentstatus",
            "RecentThemes"
        ]
    ]
)

Saved Part 3.2 Steps 1-3 to cboddick_project_stats.csv


,project,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered,Currentstatus,RecentThemes
0,baggepinnen_spectraldistances.jl,I dont see an obvious cause in the commit hist...,Yes,"Activity restarted dependency updates, followe...","Returning: CompatHelper Julia, Documenter.jl, ...",Active,Other:Automated bot contributions
1,itowns_itowns,N/A active Project,N/A,N/A,N/A,Active,Other
2,maria-korosteleva_garment-pattern-estimation,"the pre-gap work had shifted mainly to README,...",Yes,The project returned to address issue #11 by f...,"Returning: Daniel Corvesor, maria-korosteleva",Inactive,Feature development:Other
3,massimocimmino_pygfunction,The gap followed the version 1.1.0 release cyc...,Yes,Development resumed with substantial feature w...,Returning: Unknown,Active,Other:Bug fixes
4,nimble-dev_nimble,N/A active Project,N/A,N/A,N/A,Active,Other:Documentation updates
5,raysect_source,The gap followed migration work and a v0.8.1 b...,Yes,Development resumed with setup and build refac...,"New: munechika-koyo, vsnever",Active,Documentation updates:Other
6,serock3_master_thesis_qec_simulation,The repository was created for a Master's thes...,Yes,The first available post-gap WoC commit was a ...,Returning: Sebastian Holmin,Inactive,Feature development:Other
7,tf-encrypted_tf-encrypted,The pre-gap commits include compatibility and ...,Yes,The project returned for a narrow bug fix. Onl...,New: Zhicong Huang (Zico),Inactive,Other:Bug fixes
8,the-virtual-brain_tvb-root,The surrounding commits indicate repository re...,Yes,Activity resumed during GitHub publication and...,"Returning: liadomide; New: mihandrei, popaula937",Active,Other:Bug fixes
9,xtra-computing_thundergbm,The gap followed a heavy optimization and bug-...,Yes,The gap ended with two README updates. This su...,Returning: Borui Xu,Inactive,Other:Bug fixes


In [4]:
import pandas as pd

# Load the project statistics created in Steps 1-3
stats = pd.read_csv(
    "cboddick_project_stats.csv",
    sep=";"
)

notes = {

    "baggepinnen_spectraldistances.jl":
        "The project's longest inactivity gap was difficult to interpret because the surrounding commits showed routine maintenance rather than an obvious abandonment event. Activity eventually returned through dependency, compatibility, version, and build-related maintenance. Therefore, I believe it was temporary development pause rather than permanent abandonment.",

    "itowns_itowns":
        "The project maintained a steady activity pattern and had no inactivity gap. Because development continued throughout the timeline, there was no specific recovery event to interpret.",

    "maria-korosteleva_garment-pattern-estimation":
        "The project entered a long inactivity period after a lot of documentation, installation, and maintenance commits. It recovered to address an issue involving prediction naming conventions, but for only a small amount of activity. This suggests the main development effort had largely slowed or finished.",

    "massimocimmino_pygfunction":
        "The longest gap is most likely a pause between development or release cycles. The project recovered strongly with new feature development, including network-related functionality. This shows there was no abandonment, and rather a successful return to real development.",

    "nimble-dev_nimble":
        "The project maintained steady activity and had no qualifying inactivity gap. There was therefore no specific recovery period to interpret.",

    "raysect_source":
        "The exact cause of the longest gap was not obvious, but it followed migration and build-related development. Activity later resumed with refactoring, bug fixes, and additional features. The gap suggests a small break between development phases.",

    "serock3_master_thesis_qec_simulation":
        "The project's inactivity is related to the fact that it is a Master's thesis and continuation project. Only minor README work occurred after the gap, so its recovery was brief and documentation-focused. This shows that it was most likely near completion and the author returned after the gap for small document changes before submission.",

    "tf-encrypted_tf-encrypted":
        "The project showed declining activity, with compatibility and bug-fix work before and after its longest gap. It recovered for a few bug fixes, but the return was limited. The evidence suggests increasing maintenance and compatibility difficulties as development declined led to its eventual abandonment.",

    "the-virtual-brain_tvb-root":
        "The longest gap is related to repository migration and organization. Activity returned strongly through GitHub publication, repository integration, and compatibility work. The project therefore demonstrated a clear recovery, most likely due to migration of the project elsewhere.",

    "xtra-computing_thundergbm":
        "The project showed declining activity after a period of optimization and bug-fix development. Following the longest gap, only a small number of README changes occurred. This suggests that core development had largly slowed."
}


stats["Notes"] = stats["project"].map(notes)


# save to project stats

stats.to_csv(
    "cboddick_project_stats.csv",
    sep=";",
    index=False
)


print("Created the 10 reflection files.")
print("Added Notes to cboddick_project_stats.csv.")

display(
    stats[
        [
            "project",
            "Notes"
        ]
    ]
)

Created the 10 reflection files.
Added Notes to cboddick_project_stats.csv.


,project,Notes
0,baggepinnen_spectraldistances.jl,The project's longest inactivity gap was diffi...
1,itowns_itowns,The project maintained a steady activity patte...
2,maria-korosteleva_garment-pattern-estimation,The project entered a long inactivity period a...
3,massimocimmino_pygfunction,The longest gap is most likely a pause between...
4,nimble-dev_nimble,The project maintained steady activity and had...
5,raysect_source,The exact cause of the longest gap was not obv...
6,serock3_master_thesis_qec_simulation,The project's inactivity is related to the fac...
7,tf-encrypted_tf-encrypted,"The project showed declining activity, with co..."
8,the-virtual-brain_tvb-root,The longest gap is related to repository migra...
9,xtra-computing_thundergbm,The project showed declining activity after a ...


In [1]:
import pandas as pd

#some final cleanup

stats = pd.read_csv(
    "cboddick_project_stats.csv",
    sep=";"
)


# Use whichever project-name column currently exists
if "project" in stats.columns:
    project_col = "project"
elif "Project" in stats.columns:
    project_col = "Project"
else:
    raise KeyError("Could not find project column.")


# pygfunction:
# Massimo Cimmino appeared under the same email but was labeled Unknown
stats.loc[
    stats[project_col] == "massimocimmino_pygfunction",
    "WhoRecovered"
] = "Returning: Massimo Cimmino"


# raysect:
# munechika-koyo contributed before AND after the gap,
# but the email changed, so the automatic comparison treated them as new
stats.loc[
    stats[project_col] == "raysect_source",
    "WhoRecovered"
] = "Returning: munechika-koyo; New: vsnever"


# rename columns
rename_map = {
    "project": "Project",
    "number_of_commits": "ncommits",
    "number_of_authors": "nauthors",
    "min_time": "from",
    "max_time": "to",
    "number of stars": "nstars",
    "number of forks": "nforks",
    "last commit date": "lastGHCommitDate"
}

# Only rename columns that have not already been renamed
stats = stats.rename(
    columns={
        old: new
        for old, new in rename_map.items()
        if old in stats.columns
    }
)



required_columns = [
    # Part 1
    "Project",
    "ncommits",
    "nauthors",
    "from",
    "to",
    "nstars",
    "nforks",
    "lastGHCommitDate",

    # Part 2
    "LongestGapStart",
    "LongestGapEnd",
    "LongestGapLength",
    "NumCommitsAfterLastGap",
    "ActivityPattern",
    "NumberOfGapsInTimeline",

    # Part 3.1
    "BeforeThemes",
    "AfterThemes",

    # Part 3.2
    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
    "Currentstatus",
    "RecentThemes",

    # Reflection summary
    "Notes"
]



# Put columns in exact assignment order
stats = stats[required_columns]

# Save final version
stats.to_csv(
    "cboddick_project_stats.csv",
    sep=";",
    index=False
)

print("cboddick_project_stats.csv saved successfully.")
print()
print("Number of projects:", len(stats))
print("Number of columns:", len(stats.columns))
print()

print("FINAL COLUMNS:")
for col in stats.columns:
    print(" -", col)

print()
print("WHO RECOVERED CHECK:")
display(
    stats[
        [
            "Project",
            "WhoRecovered"
        ]
    ]
)

print()
print("FINAL STATS PREVIEW:")
display(stats)

cboddick_project_stats.csv saved successfully.

Number of projects: 10
Number of columns: 23

FINAL COLUMNS:
 - Project
 - ncommits
 - nauthors
 - from
 - to
 - nstars
 - nforks
 - lastGHCommitDate
 - LongestGapStart
 - LongestGapEnd
 - LongestGapLength
 - NumCommitsAfterLastGap
 - ActivityPattern
 - NumberOfGapsInTimeline
 - BeforeThemes
 - AfterThemes
 - HypothesizedGapReason
 - HasRecovered
 - WhyRecovered
 - WhoRecovered
 - Currentstatus
 - RecentThemes
 - Notes

WHO RECOVERED CHECK:


,Project,WhoRecovered
0,baggepinnen_spectraldistances.jl,"Returning: CompatHelper Julia, Documenter.jl, ..."
1,itowns_itowns,NaN
2,maria-korosteleva_garment-pattern-estimation,"Returning: Daniel Corvesor, maria-korosteleva"
3,massimocimmino_pygfunction,Returning: Massimo Cimmino
4,nimble-dev_nimble,NaN
5,raysect_source,Returning: munechika-koyo; New: vsnever
6,serock3_master_thesis_qec_simulation,Returning: Sebastian Holmin
7,tf-encrypted_tf-encrypted,New: Zhicong Huang (Zico)
8,the-virtual-brain_tvb-root,"Returning: liadomide; New: mihandrei, popaula937"
9,xtra-computing_thundergbm,Returning: Borui Xu



FINAL STATS PREVIEW:


,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,...,NumberOfGapsInTimeline,BeforeThemes,AfterThemes,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered,Currentstatus,RecentThemes,Notes
0,baggepinnen_spectraldistances.jl,759,7,1568771532,1776126985,49,5,2026-01-15,2023-02,2025-06,...,3,Dependency updates:Feature development,Dependency updates:Release,I dont see an obvious cause in the commit hist...,Yes,"Activity restarted dependency updates, followe...","Returning: CompatHelper Julia, Documenter.jl, ...",Active,Other:Automated bot contributions,The project's longest inactivity gap was diffi...
1,itowns_itowns,9190,142,1443101686,1762275571,1300,317,2026-09-14,NaN,NaN,...,0,N/A active Project,N/A active Project,N/A active Project,NaN,NaN,NaN,Active,Other,The project maintained a steady activity patte...
2,maria-korosteleva_garment-pattern-estimation,1378,11,1582260169,1736211317,181,21,2024-12-23,2023-12,2024-11,...,3,Documentation updates:Dependency updates,Bug fixes:Other,"the pre-gap work had shifted mainly to README,...",Yes,The project returned to address issue #11 by f...,"Returning: Daniel Corvesor, maria-korosteleva",Inactive,Feature development:Other,The project entered a long inactivity period a...
3,massimocimmino_pygfunction,1385,22,1502741635,1771443588,72,26,2025-08-04,2018-04,2018-10,...,7,Bug fixes:Documentation updates,Feature development:Documentation updates,The gap followed the version 1.1.0 release cyc...,Yes,Development resumed with substantial feature w...,Returning: Massimo Cimmino,Active,Other:Bug fixes,The longest gap is most likely a pause between...
4,nimble-dev_nimble,9730,73,1402585122,1762381653,198,27,2026-08-31,NaN,NaN,...,0,N/A active Project,N/A active Project,N/A active Project,NaN,NaN,NaN,Active,Other:Documentation updates,The project maintained steady activity and had...
5,raysect_source,2366,33,1376432727,1758534567,112,25,2025-08-25,2024-03,2024-09,...,4,Feature development:Other,Other:Dependency updates,The gap followed migration work and a v0.8.1 b...,Yes,Development resumed with setup and build refac...,Returning: munechika-koyo; New: vsnever,Active,Documentation updates:Other,The exact cause of the longest gap was not obv...
6,serock3_master_thesis_qec_simulation,496,6,1611075452,1639215098,5,0,2024-09-07,2021-11,2021-11,...,0,Documentation updates:Feature development,Documentation updates,The repository was created for a Master's thes...,Yes,The first available post-gap WoC commit was a ...,Returning: Sebastian Holmin,Inactive,Feature development:Other,The project's inactivity is related to the fac...
7,tf-encrypted_tf-encrypted,3165,49,1521656591,1727242358,1200,211,2024-09-25,2023-11,2024-08,...,4,Bug fixes:Documentation updates,Bug fixes,The pre-gap commits include compatibility and ...,Yes,The project returned for a narrow bug fix. Onl...,New: Zhicong Huang (Zico),Inactive,Other:Bug fixes,"The project showed declining activity, with co..."
8,the-virtual-brain_tvb-root,10306,136,1401877909,1776180337,190,151,2026-09-23,2018-07,2018-11,...,1,Documentation updates:Other,Release:Bug fixes,The surrounding commits indicate repository re...,Yes,Activity resumed during GitHub publication and...,"Returning: liadomide; New: mihandrei, popaula937",Active,Other:Bug fixes,The longest gap is related to repository migra...
9,xtra-computing_thundergbm,683,24,1453180646,1742375195,715,88,2025-03-19,2024-02,2025-02,...,7,Feature development:Bug fixes,Documentation updates,The gap followed a heavy optimization and bug-...,Yes,The gap ended with two README updates. This su...,Returning: Borui Xu,Inactive,Other:Bug fixes,The project showed declining activity after a ...
